In [16]:
from pathlib import Path
import pymupdf
from IPython.display import Image, display

In [24]:
DATA_DIR = Path("../data/raw")
list(DATA_DIR.glob("*.pdf"))

[WindowsPath('../data/raw/microsoft_report.pdf'),
 WindowsPath('../data/raw/test.pdf'),
 WindowsPath('../data/raw/worldbank_report.pdf')]

In [25]:
pdf_path = next(DATA_DIR.glob("*test*")) # next() pour récupérer la valeur de l'objet créer par glob()

print(pdf_path)

..\data\raw\test.pdf


In [26]:
doc = pymupdf.open(pdf_path)

print(f"Nombre de pages : {len(doc)}")
print(f"Métadonnées : {doc.metadata}")

Nombre de pages : 2
Métadonnées : {'format': 'PDF 1.4', 'title': 'Document sans titre', 'author': '', 'subject': '', 'keywords': '', 'creator': '', 'producer': 'Skia/PDF m155 Google Docs Renderer', 'creationDate': '', 'modDate': '', 'trapped': '', 'encryption': None}


In [ ]:
for numero, page in enumerate(doc):
    print(f"\n--- Page {numero + 1} ---")

    ### Texte
    texte = page.get_text("text")
    print("Nombre de caractères :", len(texte))

    ### Tableau
    resultats = page.find_tables()
    print("Nombre de tableaux détectés :", len(resultats.tables))

    for i, tableau in enumerate(resultats.tables):
        print(
            f"  Tableau {i} : "
            f"{tableau.row_count} lignes, "
            f"{tableau.col_count} colonnes")

    ### Image
    image_list = page.get_images()
    print("Nombre d'image détectées :", len(image_list))

    for image_index, img in enumerate(image_list, start=1):
        xref = img[0]
        largeur = img[2]
        hauteur = img[3]

        positions = page.get_image_rects(xref)

        print(f"\nImage {image_index}")
        print("  xref :", xref)
        print("  dimensions :", largeur, "x", hauteur)
        print("  positions :", positions)


--- Page 1 ---
Nombre de caractères : 1195
Nombre de tableaux détectés : 1
  Tableau 0 : 20 lignes, 3 colonnes
Nombre d'image détectées : 0

--- Page 2 ---
Nombre de caractères : 14
Nombre de tableaux détectés : 0
Nombre d'image détectées : 2

Image 1
  xref : 9
  dimensions : 200 x 200
  positions : [Rect(73.5, 66.9749755859375, 215.25, 208.7249755859375)]

Image 2
  xref : 11
  dimensions : 855 x 535
  positions : [Rect(73.5, 278.3123779296875, 525.0, 560.3123779296875)]


Doc PDF -> Détection des éléments :

-> Texte -> Chunking textuel

-> Tableau -> Chunking tabulaire

-> Images, graphiques -> OCR ou modèle de vision

-> Index vectoriel

-> Réponse aux sources

-> Surlignage par bbox

# Tableau

In [79]:
tableaux_document = [] # Liste de tableaux par document

for numero, page in enumerate(doc):
    #print(f"\n--- Page {numero + 1} ---")

    resultats = page.find_tables()
    #print("Nombre de tableaux détectés :", len(resultats.tables))

    for i, tableau in enumerate(resultats.tables):

            # Extraction des coordonnées du tableau

        bbox_tableau = list(tableau.bbox)

        geometrie_entete = tableau.rows[0]
        bbox_entete = list(geometrie_entete.bbox)

        donnees = tableau.extract() # Extraction du contenu du tableau. Les donnees deviennent une liste contenant toutes les lignes 

        if donnees:
            titres_colonnes = donnees[0]

            lignes_donnees = donnees[1:]

            if lignes_donnees:

                lignes_structurees = []

                for numero_ligne, ligne in enumerate(lignes_donnees,start=1): # start=1 pour correspondre aux index de tableau.rows car tableau.rows[0] contient l'en-tête

                    # Association des titres avec les valeurs
                    donnees_ligne = dict(zip(titres_colonnes, ligne)) # # zip() associe les éléments qui ont la même position
                    
                    # Géométrie ligne actuelle
                    geometrie_ligne = tableau.rows[numero_ligne]

                    # Bbox de la ligne complète
                    bbox_ligne = list(geometrie_ligne.bbox)
                    
                    # Liste des cellules de la ligne actuelle
                    cellules_structurees = []

                    for numero_colonne, titre in enumerate(titres_colonnes):

                        valeur = ligne[numero_colonne]

                        bbox_cellule = geometrie_ligne.cells[numero_colonne]

                        cellule_structuree = {
                            "colonne": titre,
                            "valeur": valeur,
                            "bbox": (
                                list(bbox_cellule) 
                                if bbox_cellule is not None
                                else None
                            )
                        }

                        cellules_structurees.append(cellule_structuree)

                    # Création de la ligne après avoir traité ses cellules
                    ligne_structuree = {
                        "data": donnees_ligne,
                        "bbox": bbox_ligne,
                        "cellules": cellules_structurees
                    }

                    # Ajout de la ligne terminée au tableau
                    lignes_structurees.append(ligne_structuree)
                
                # Construction d'un tableau complet
                tableau_structure = {
                    "type": "tableau",
                    "page": numero +1,
                    "bbox": bbox_tableau,
                    "colonnes": titres_colonnes,
                    "bbox_entete": bbox_entete,
                    "lignes": lignes_structurees

                }

                # Ajout du tableau à la liste du docume,nt
                tableaux_document.append(tableau_structure)


print(
    "Nombre total de tableaux :",
    len(tableaux_document)
)

premiere_ligne_structuree = (
    tableaux_document[0]["lignes"][0]
)

print(
    "Données de la première ligne :",
    premiere_ligne_structuree["data"]
)

print(
    "Bbox de la première ligne :",
    premiere_ligne_structuree["bbox"]
)

print(
    "Nombre de cellules :",
    len(premiere_ligne_structuree["cellules"])
)
    

Nombre total de tableaux : 1
Données de la première ligne : {'a': '1', 'b': '1', 'c': '1'}
Bbox de la première ligne : [72.5, 255.5, 297.5, 271.5]
Nombre de cellules : 3


In [91]:
print(tableaux_document[0]['type'])
print(tableaux_document[0]['page'])
print(tableaux_document[0]['bbox'])
print(tableaux_document[0]['colonnes'])
print(tableaux_document[0]['bbox_entete'])
print(tableaux_document[0]['lignes'])

tableau
1
[72.5, 238.5, 297.5, 568.5]
['a', 'b', 'c']
[72.5, 238.5, 297.5, 255.5]
[{'data': {'a': '1', 'b': '1', 'c': '1'}, 'bbox': [72.5, 255.5, 297.5, 271.5], 'cellules': [{'colonne': 'a', 'valeur': '1', 'bbox': [72.5, 255.5, 147.5, 271.5]}, {'colonne': 'b', 'valeur': '1', 'bbox': [147.5, 255.5, 222.5, 271.5]}, {'colonne': 'c', 'valeur': '1', 'bbox': [222.5, 255.5, 297.5, 271.5]}]}, {'data': {'a': '2', 'b': '2', 'c': '2'}, 'bbox': [72.5, 271.5, 297.5, 288.5], 'cellules': [{'colonne': 'a', 'valeur': '2', 'bbox': [72.5, 271.5, 147.5, 288.5]}, {'colonne': 'b', 'valeur': '2', 'bbox': [147.5, 271.5, 222.5, 288.5]}, {'colonne': 'c', 'valeur': '2', 'bbox': [222.5, 271.5, 297.5, 288.5]}]}, {'data': {'a': '3', 'b': '3', 'c': '3'}, 'bbox': [72.5, 288.5, 297.5, 304.5], 'cellules': [{'colonne': 'a', 'valeur': '3', 'bbox': [72.5, 288.5, 147.5, 304.5]}, {'colonne': 'b', 'valeur': '3', 'bbox': [147.5, 288.5, 222.5, 304.5]}, {'colonne': 'c', 'valeur': '3', 'bbox': [222.5, 288.5, 297.5, 304.5]}]}, {'

In [92]:
# Première ligne de valeur du tableau
premiere_ligne = tableaux_document[0]["lignes"][0]
print("Données :", premiere_ligne["data"])
print("Bbox :", premiere_ligne["bbox"])
print("Cellules :", premiere_ligne["cellules"])

Données : {'a': '1', 'b': '1', 'c': '1'}
Bbox : [72.5, 255.5, 297.5, 271.5]
Cellules : [{'colonne': 'a', 'valeur': '1', 'bbox': [72.5, 255.5, 147.5, 271.5]}, {'colonne': 'b', 'valeur': '1', 'bbox': [147.5, 255.5, 222.5, 271.5]}, {'colonne': 'c', 'valeur': '1', 'bbox': [222.5, 255.5, 297.5, 271.5]}]


# Texte

In [ ]:
for numero, page in enumerate(doc):
    blocs = page.get_text("blocks", sort=True)

    print(blocs)### ###rrvergzerrvdrgERDFGBFGNFC CV SGS###737

[(272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039, 'TEST \n', 0, 0), (72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539, 'Qu’est-ce que la génération à enrichissement contextuel ? \n', 1, 0), (72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844, 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de \n', 2, 0), (72.0, 124.99349975585938, 509.201171875, 133.93099975585938, 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner \n', 3, 0), (72.0, 139.71224975585938, 525.177734375, 148.64974975585938, 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et \n', 4, 0), (72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938, 'utilisent des milliards de paramètres pour générer des résultats originaux pour des tâches telles que répon

In [ ]:
blocs_texte_bruts = []

for numero, page in enumerate(doc):

    # Récupération du texte sous forme de blocs avec coordonnées, dans l'ordre de lecture
    blocs = page.get_text("blocks", sort=True)
    
    for bloc in blocs:
        x0, y0, x1, y1, contenu, numero_bloc, type_bloc = bloc

        if type_bloc == 0 and contenu.strip(): # == 0 pour conserver uniquement les blocs textuels / .strip() élimine les blocs vides
            bloc_texte = {
                "page": numero +1,
                "content": contenu.strip(),
                "bbox": [x0, y0, x1, y1]
            }

            blocs_texte_bruts.append(bloc_texte)

In [103]:
print(blocs_texte_bruts)

[{'page': 1, 'content': 'TEST', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}, {'page': 1, 'content': 'Qu’est-ce que la génération à enrichissement contextuel ?', 'bbox': [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]}, {'page': 1, 'content': 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de', 'bbox': [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]}, {'page': 1, 'content': 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner', 'bbox': [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]}, {'page': 1, 'content': 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et', 'bbox': [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]}, {'page': 1, 'content': 'utilisent des milliards de paramètr

## Elimination des blocs textuels appartenant au tableau

In [111]:
blocs_texte_hors_tableaux = []

# Récupération bbox blocs
for bloc in blocs_texte_bruts:

    # Transformation bbox blocs texte en rectangle PyMuPDF
    rectangle_bloc = pymupdf.Rect(bloc["bbox"])

    # Variable appartenance au tableau
    bloc_dans_tableau = False

    # Comparaison des mêmes pages
    for tableau in tableaux_document:
        if tableau["page"] != bloc["page"]:
            continue
        
        # Transformation bbox tableau en rectangle PyMuPDF
        rectangle_tableau = pymupdf.Rect(
            tableau["bbox"]
        )

        # Vérification si intersection entre les deux rectangles
        if rectangle_tableau.intersects(rectangle_bloc):
            bloc_dans_tableau = True
            break
    
    # Conservation du bloc s'il n'est pas dans le tableau
    if not bloc_dans_tableau:
        blocs_texte_hors_tableaux.append(
            bloc
        )

In [112]:
print(
    "Blocs avant filtrage :",
    len(blocs_texte_bruts)
)

print(
    "Blocs après filtrage :",
    len(blocs_texte_hors_tableaux)
)

Blocs avant filtrage : 30
Blocs après filtrage : 10


In [113]:
for bloc in blocs_texte_hors_tableaux:
    print("\nPage :", bloc["page"])
    print("Bbox :", bloc["bbox"])
    print("Texte :", repr(bloc["content"]))


Page : 1
Bbox : [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]
Texte : 'TEST'

Page : 1
Bbox : [72.0, 87.3157730102539, 280.3662414550781, 96.2532730102539]
Texte : 'Qu’est-ce que la génération à enrichissement contextuel ?'

Page : 1
Bbox : [72.0, 110.27476501464844, 505.14263916015625, 119.21226501464844]
Texte : 'La génération à enrichissement contextuel (RAG) est le processus consistant à optimiser le résultat d’un grand modèle de'

Page : 1
Bbox : [72.0, 124.99349975585938, 509.201171875, 133.93099975585938]
Texte : 'langage. Elle fait donc appel à une base de connaissances fiable externe aux sources de données utilisées pour l’entraîner'

Page : 1
Bbox : [72.0, 139.71224975585938, 525.177734375, 148.64974975585938]
Texte : 'avant de générer une réponse. Les grands modèles de langage (LLM) sont entraînés avec d’importants volumes de données et'

Page : 1
Bbox : [72.0, 154.43099975585938, 522.0401611328125, 163.36849975585938]
Texte : 'utilisent des

In [117]:
textes_document = []

for bloc in blocs_texte_hors_tableaux:
    texte_structure = {
        "type": "texte",
        "page": bloc["page"],
        "content": bloc["content"],
        "bbox": bloc["bbox"]
    }

    textes_document.append(
        texte_structure
    )

In [118]:
print(
    "Nombre de blocs de texte :",
    len(textes_document)
)

print(
    "Premier bloc :",
    textes_document[0]
)

Nombre de blocs de texte : 10
Premier bloc : {'type': 'texte', 'page': 1, 'content': 'TEST', 'bbox': [272.09112548828125, 50.260597229003906, 328.74444580078125, 72.6043472290039]}
